# Finnhub - Khám phá sàn chứng khoán FinHub API

Notebook này giúp bạn khám phá nhanh thư viện `finnhub-python`:
- Kiểm tra kết nối API qua biến môi trường
- Kiểm tra dữ liệu real-time qua websocket
- Lấy dữ liệu nến từ quá khứ
- Lấy tin tức chung của sàn chứng khoán

## 1. Cấu hình

In [2]:
import sys
import os
import asyncio
import finnhub
from datetime import datetime
from IPython.display import clear_output, display

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.config import Config
from src.streamer import finnhub_streamer

In [3]:
# Kiểm tra key
if not Config.FINNHUB_API_KEY:
    print("Error: FINNHUB_API_KEY is not set in the environment variables.")
    sys.exit(1)
else:
    print("FINNHUB_API_KEY: ", Config.FINNHUB_API_KEY[0:4] + "...")  # In một phần của key để xác nhận

FINNHUB_API_KEY:  d7f7...


## 2. Giao dịch - Cập nhật giá cuối cùng

Finhub có hỗ trợ cho bản free Websocket để lấy giá liên tục từ chứng khoán thị trường Hoa Kỳ

In [ ]:
symbols_to_track = ["AAPL", "BINANCE:BTCUSDT", "OANDA:EUR_USD"]
api_key = Config.FINNHUB_API_KEY 

# Khởi tạo dữ liệu trống
latest_prices = {symbol: {"price": "Đang đợi...", "volume": "0", "time": "--:--:--"} for symbol in symbols_to_track}

async def run_tracker():
    print("🚀 Đang kết nối server Finnhub...")
    
    try:
        async for trades in finnhub_streamer(api_key, symbols_to_track):
            for trade in trades:
                symbol = trade['s']
                # Định dạng lại thời gian (Giờ:Phút:Giây)
                dt_object = datetime.fromtimestamp(trade['t'] / 1000.0)
                time_str = dt_object.strftime('%H:%M:%S')
                
                # Cập nhật thông tin vào dictionary
                latest_prices[symbol] = {
                    "price": f"{trade['p']:,.2f}",
                    "volume": f"{trade['v']:.4f}",
                    "time": time_str
                }
            
            # Xóa cell và in lại bảng mới
            clear_output(wait=True)
            print(f"📊 BẢNG GIÁ REAL-TIME ({datetime.now().strftime('%d/%m/%Y')})")
            print("-" * 65)
            print(f"{'Mã':<18} | {'Giá (USD)':>12} | {'KL':>12} | {'Thời gian'}")
            print("-" * 65)
            
            for sym, info in latest_prices.items():
                # ĐÃ SỬA LỖI: Truy cập đúng info['volume']
                print(f"{sym:<18} | {info['price']:>12} | {info['volume']:>12} | {info['time']}")
            
            print("-" * 65)
            print("💡 Mẹo: Nhấn nút Stop (Interupt) để dừng cập nhật.")

    except asyncio.CancelledError:
        print("\n⏹️ Đã dừng Task thành công.")
    except Exception as e:
        print(f"\n❌ Có lỗi xảy ra: {e}")

stream_task = asyncio.create_task(run_tracker())

📊 BẢNG GIÁ REAL-TIME (17/04/2026)
-----------------------------------------------------------------
Mã                 |    Giá (USD) |           KL | Thời gian
-----------------------------------------------------------------
AAPL               |  Đang đợi... |            0 | --:--:--
BINANCE:BTCUSDT    |    74,899.37 |       0.0031 | 08:20:46
OANDA:EUR_USD      |         1.18 |       0.0000 | 08:20:45
-----------------------------------------------------------------
💡 Mẹo: Nhấn nút Stop (Interupt) để dừng cập nhật.

⏹️ Đã dừng Task thành công.


Nếu muốn dừng, bạn chạy lệnh dưới đây

In [5]:
stream_task.cancel() # Dừng task nếu cần thiết

True

Dưới đây là phần kiểm tra độ trễ của từng trading

In [ ]:
# Ngưỡng trễ (5 phút)
LATENCY_THRESHOLD = 300 

# Khởi tạo stats cho từng mã trong danh sách theo dõi
diagnostic_stats = {
    symbol: {
        "total": 0,
        "late_count": 0,
        "max_delay": 0,
    } for symbol in symbols_to_track
}

# Danh sách log chung cho các giao dịch bị trễ
late_alerts_log = []

async def run_multi_diagnostic():
    print("🔍 Đang khởi tạo hệ thống giám sát đa luồng...")
    
    try:
        async for trades in finnhub_streamer(api_key, symbols_to_track):
            current_time_ms = datetime.now().timestamp() * 1000
            
            for trade in trades:
                sym = trade['s']
                event_time_ms = trade['t']
                delay_sec = (current_time_ms - event_time_ms) / 1000
                
                if sym in diagnostic_stats:
                    # 1. Cập nhật thống kê cơ bản
                    diagnostic_stats[sym]["total"] += 1
                    if delay_sec > diagnostic_stats[sym]["max_delay"]:
                        diagnostic_stats[sym]["max_delay"] = delay_sec
                    
                    # 2. Kiểm tra nếu trễ quá 5 phút
                    if delay_sec > LATENCY_THRESHOLD:
                        diagnostic_stats[sym]["late_count"] += 1
                        
                        # Ghi log cảnh báo
                        alert = {
                            "symbol": sym,
                            "price": trade['p'],
                            "delay": f"{delay_sec/60:.1f}m",
                            "time": datetime.fromtimestamp(event_time_ms/1000).strftime('%H:%M:%S')
                        }
                        late_alerts_log.insert(0, alert)
                        if len(late_alerts_log) > 8: late_alerts_log.pop()

            # --- HIỂN THỊ GIAO DIỆN ---
            clear_output(wait=True)
            print(f"🕵️ GIÁM SÁT ĐỘ TRỄ HỆ THỐNG ({datetime.now().strftime('%H:%M:%S')})")
            print("=" * 75)
            print(f"{'Mã Chứng Khoán':<18} | {'Tổng Trade':>10} | {'Trễ >5p':>10} | {'Trễ Max (Phút)':>15}")
            print("-" * 75)
            
            for sym, stats in diagnostic_stats.items():
                max_min = stats['max_delay'] / 60
                # Đánh dấu cảnh báo nếu có trade trễ
                status_icon = "❌" if stats['late_count'] > 0 else "✅"
                
                print(f"{status_icon} {sym:<15} | {stats['total']:>10} | {stats['late_count']:>10} | {max_min:>15.2f}")
            
            print("=" * 75)
            if late_alerts_log:
                print("⚠️ CẢNH BÁO GIAO DỊCH ĐẾN MUỘN GẦN ĐÂY:")
                for a in late_alerts_log:
                    print(f"• [{a['time']}] {a['symbol']}: Giá {a['price']} (Trễ {a['delay']})")
            else:
                print("✨ Tín hiệu ổn định: Chưa phát hiện dữ liệu cũ lọt vào luồng thực tế.")

    except asyncio.CancelledError:
        print("\n⏹️ Đã dừng giám sát.")

# Kích hoạt task
diag_task = asyncio.create_task(run_multi_diagnostic())

🕵️ GIÁM SÁT ĐỘ TRỄ HỆ THỐNG (02:06:40)
Mã Chứng Khoán     | Tổng Trade |    Trễ >5p |  Trễ Max (Phút)
---------------------------------------------------------------------------
✅ AAPL            |         24 |          0 |            1.82
✅ BINANCE:BTCUSDT |       3691 |          0 |            2.46
✅ AMZN            |         38 |          0 |            1.76
✨ Tín hiệu ổn định: Chưa phát hiện dữ liệu cũ lọt vào luồng thực tế.

⏹️ Đã dừng giám sát.


In [35]:
diag_task.cancel() # Dừng task chẩn đoán nếu không cần thiết nữa

True

Hiện tại dữ liệu được lấy liên tục từ finnhub không được đúng theo trình tự, có một vài trading (phần lớn từ crypto) có độ trễ rơi và khoảng 2-4 phút mới tới nơi. Nên nến sẽ được tính min là 5 phút để tránh biểu đồ rơi rớt dữ liệu.

## 3. Dữ liệu nến từ quá khứ

Do finnhub đã khóa lấy dữ liệu nến quá khứ cho free, để bù đắp chuyện này, chúng ta sẽ sử dụng thư viện yfinance để bù lại và nối đuôi với finnhub thực tế sau